# Introduction to Machine Learning — Class 8
## Ensemble Methods & Model-Centric vs Data-Centric ML

**Input dataset:** `adult_classification_v1.csv`  
**Problem:** Binary classification  

---

### Learning goals
- Understand why ensembles often outperform single models
- Apply basic ensemble methods
- Compare model-centric and data-centric strategies
- Reflect on long-term ML system performance

> **Key idea:** Improving a system is not always about changing the model.

<div style="margin-left: 2em; font-size: 0.9em; font-style: italic;">
An AI language model (ChatGPT by OpenAI) was used to support the creation of practical class materials. 
All arguments, outputs, and final wording were critically reviewed, edited, and validated by the author 
prior to use with students.
</div>

## 0. Data context

You will continue using the **same dataset** as in previous sessions.

The goal is **not** to achieve the best possible score, but to compare strategies:
- improving the model
- improving the data


In [ ]:
DATA_PATH = "adult_classification_v1.csv"


## 1. Imports


In [ ]:
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold


In [ ]:
# ✅ Auto-check
import importlib
_required = ["numpy", "pandas", "sklearn"]
missing = [m for m in _required if importlib.util.find_spec(m) is None]
assert not missing, f"Missing packages: {missing}"
print("Auto-check passed.")


## 2. Load data

### Tasks
- Load the dataset
- Separate features and target
- Reflect on the need of data scaling


In [ ]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["income"])
y = df["income"]

print(f"Shape: {X.shape}")
df.head()


## 3. Baseline model

### Task
- Evaluate a simple baseline model using cross-validation

**Reflection**
- How strong is the baseline?

> A single Decision Tree tends to overfit the training data, producing high variance across folds. Its accuracy gives us a reference point: any ensemble method should ideally improve on this score and, just as importantly, reduce the standard deviation across folds (i.e. be more stable). Note that tree-based models do **not** require feature scaling, but we include `StandardScaler` in the pipeline so the setup is consistent and ready if we later swap in a distance-based estimator.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42)),
])

baseline_scores = cross_val_score(baseline_pipe, X, y, cv=cv, scoring="accuracy")
print(f"Baseline (Decision Tree pipeline): {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}")


## 4. Bagging (Bootstrap Aggregating)

Bagging reduces variance by averaging multiple models.

### Tasks
- Train a Bagging classifier
- Compare performance with baseline

**Reflection**
- Why does bagging help?

> Bagging trains each base learner on a different bootstrap sample of the data and then aggregates their predictions (majority vote for classification). Because individual Decision Trees are high-variance models, averaging many of them smooths out the overfitting, leading to lower variance without significantly increasing bias. The result is usually a higher and more stable cross-validation score compared to a single tree.


In [ ]:
bagging_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=50,
        random_state=42,
    )),
])

bagging_scores = cross_val_score(bagging_pipe, X, y, cv=cv, scoring="accuracy")
print(f"Bagging: {bagging_scores.mean():.4f} ± {bagging_scores.std():.4f}")


## 5. Random Forest

Random Forest combines:
- bagging
- feature randomness

### Tasks
- Train a Random Forest classifier
- Compare performance

**Reflection**
- What additional randomness is introduced?

> On top of bootstrap sampling (like Bagging), Random Forest also randomly selects a **subset of features** at each split. This de-correlates the individual trees: in plain Bagging the same dominant features tend to appear at the top of every tree, making the ensemble's members similar. By forcing each split to consider only √p (classification default) features, the trees become more diverse, and their average prediction improves further.


In [ ]:
rf_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
    )),
])

rf_scores = cross_val_score(rf_pipe, X, y, cv=cv, scoring="accuracy")
print(f"Random Forest: {rf_scores.mean():.4f} ± {rf_scores.std():.4f}")


## 6. Boosting

Boosting focuses on:
- correcting previous errors

### Tasks
- Train a Gradient Boosting classifier
- Compare performance

**Reflection**
- How is boosting different from bagging?

> Bagging trains models **in parallel** on independent bootstrap samples; Boosting trains them **sequentially**, where each new model focuses on the mistakes of the previous ones. Bagging mainly reduces **variance**, while Boosting primarily reduces **bias** (and can also reduce variance). Gradient Boosting fits each new tree to the negative gradient of the loss function, effectively correcting residual errors. The trade-off is that Boosting is more prone to overfitting if not properly regularised (e.g. via `learning_rate`, `max_depth`, or early stopping).


In [ ]:
gb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
    )),
])

gb_scores = cross_val_score(gb_pipe, X, y, cv=cv, scoring="accuracy")
print(f"Gradient Boosting: {gb_scores.mean():.4f} ± {gb_scores.std():.4f}")


## 7. Model-centric vs Data-centric ML

### Tasks
- Compare gains from:
  - changing the model
  - changing the data (conceptual)

**Reflection**
- When is data-centric ML preferable?

> **Model-centric ML** focuses on trying different algorithms, tuning hyperparameters, and engineering model architectures while keeping the data fixed. **Data-centric ML** keeps the model relatively fixed and instead invests effort in improving data quality — cleaning labels, removing noise, fixing inconsistencies, collecting more representative samples, and improving feature definitions.

> Data-centric approaches are preferable when:
> - The data contains labelling errors or ambiguous examples (noisy labels hurt all models equally).
> - Performance has plateaued across multiple model families, suggesting the bottleneck is data quality, not model capacity.
> - The dataset is small — a cleaner, more curated dataset can outperform a larger but noisy one.
> - The model will be deployed in production, where systematic data pipelines and monitoring matter more than marginal algorithmic gains.


## 8. Critical reflection

Answer in Markdown:
- Why do ensembles often perform better?
- What are the costs of ensemble methods?
- When should we stop improving models and improve data instead?
- What is the meaning or definition of quality in ML models?


### Reflection

**Why do ensembles often perform better?**

> Ensembles combine multiple weak or moderate learners, reducing the risk of any single model's idiosyncratic errors dominating the prediction. By averaging (bagging) or sequentially correcting (boosting), they achieve lower variance, lower bias, or both, compared to individual models.

**What are the costs of ensemble methods?**

> - **Computational cost:** Training and predicting with many models is slower and uses more memory.
> - **Interpretability:** A single Decision Tree is easy to explain; a forest of 100 trees or a chain of 100 boosted stumps is not.
> - **Diminishing returns:** After a certain number of estimators, additional models contribute very little improvement.
> - **Hyperparameter complexity:** More knobs to tune (number of estimators, learning rate, max depth, subsampling ratio, etc.).

**When should we stop improving models and improve data instead?**

> When multiple well-tuned model families converge to similar scores, the performance ceiling is likely imposed by the data, not the algorithm. At that point, effort is better spent on cleaning labels, collecting more representative samples, engineering better features, or fixing data-pipeline bugs. In practice, improving data quality almost always yields more sustainable long-term gains than model swapping.

**What is the meaning or definition of quality in ML models?**

> Model quality is multi-dimensional. It includes predictive performance (accuracy, F1, AUC), but also:
> - **Robustness:** Does the model perform consistently across different subgroups and over time?
> - **Fairness:** Does it treat all demographic groups equitably?
> - **Interpretability:** Can stakeholders understand and trust the predictions?
> - **Efficiency:** Is it fast and cheap enough to deploy and maintain?
> - **Reliability of the data pipeline:** A model is only as good as the data it receives in production.
